# Athena Advisor: Ablations

Runs the `ablations` sweep: the five ablated models on StackSample, 5 seeds each (25 jobs). All logic lives in the `athena_advisor` package; this notebook only runs
its scripts.

**Before you run** (notebook settings, right-hand panel):
1. **Accelerator:** GPU (T4 x2 or P100).
2. **Internet:** on (to clone the repository and download the encoder).
3. **Input:** add the output of notebook 01 (it contains `athena_data.zip`).
4. **To continue an unfinished sweep:** also add the output of an earlier run of this notebook. Its results
   are restored first, so finished jobs are skipped.

Jobs that already have a run record are skipped, and no new job starts after 11 hours, so the notebook ends
inside Kaggle's 12-hour limit. Run it again (with this run's output attached) until every job is done.

## 1. Environment

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU attached: training will be very slow"

## 2. Get the code

In [ ]:
import os
import subprocess

REPO = "https://github.com/pxsha23/ProjectAthena.git"
DEST = "/kaggle/working/ProjectAthena"
ML = f"{DEST}/ml"
# Never wait for a username prompt: fail fast instead.
os.environ["GIT_TERMINAL_PROMPT"] = "0"


def clone(url: str) -> bool:
    # Output is captured, not printed, so a URL that carries the token is never shown.
    return subprocess.run(["git", "clone", "--depth", "1", url, DEST], capture_output=True).returncode == 0


if not os.path.isdir(DEST):
    if not clone(REPO):
        from kaggle_secrets import UserSecretsClient

        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        if not clone(REPO.replace("https://", f"https://x-access-token:{token}@")):
            raise RuntimeError("Clone failed: check the GITHUB_TOKEN secret and that Internet is on.")
        subprocess.run(["git", "-C", DEST, "remote", "set-url", "origin", REPO], check=True)
        del token

!git -C {DEST} log -1 --format="Commit %h, %cd: %s"

## 3. Install

The package is installed without its dependencies so Kaggle's GPU build of PyTorch stays in place.

In [ ]:
%pip install -q -e {ML} --no-deps
%pip install -q setfit optimum onnx onnxruntime

In [ ]:
os.environ["ATHENA_ML_ROOT"] = ML  # relative paths in the configs resolve to the ml/ folder
os.chdir(ML)
!python scripts/check_environment.py

## 4. Restore the data and earlier results

Finds the processed data (notebook 01's `athena_data.zip`, or an unzipped `data/processed` folder) and any
earlier results zip under `/kaggle/input`, and copies them into `ml/`.

In [ ]:
!python scripts/kaggle_restore.py --input-root /kaggle/input

## 5. Run the `ablations` sweep

In [ ]:
!python scripts/sweep.py configs/sweeps/ablations.yaml --time-budget-hours 11

## 6. Results so far

In [ ]:
!python scripts/results_table.py results

## 7. Package the results

Writes `/kaggle/working/athena_results_ablations.zip` with the run records, predictions and training logs (no
checkpoints). Download it from the Output panel, or attach this notebook's output as an input to continue the
sweep or to run the evaluation notebooks.

In [ ]:
!python scripts/pack_results.py --out /kaggle/working/athena_results_ablations.zip